# 01 · Data Cleaning

Cleans each of the ten raw FRED series in `data/raw/` on its own and writes the results to `data/cleaned/`, ready to load into MySQL.

Every series gets the same treatment: rename the columns, convert dates, and trim to Jan 2007 through May 2026 (the extra year before 2008 is there so year-over-year changes can be calculated from the start of the study window). Gas prices, mortgage rates, and the fed funds rate are weekly or daily, so they're averaged into months. CPI and unemployment are both missing October 2025, which wasn't collected during the federal government shutdown, so that month is filled by linear interpolation.

In [1]:
import pandas as pd

In [2]:
# read in consumer_sentiment.csv
consumer_sentiment = pd.read_csv('../data/raw/consumer_sentiment.csv')

# rename columns
consumer_sentiment.columns = ["Date", "ConsumerSentiment"]

# convert to datetime
consumer_sentiment["Date"] = pd.to_datetime(consumer_sentiment["Date"])

# filter dates (project from Jan 2007 - May 2026)
consumer_sentiment = consumer_sentiment[(consumer_sentiment["Date"] >= "2007-01-01") &
                                        (consumer_sentiment["Date"] <= "2026-05-01")]

# check for duplicates and missing values
print("Duplicate rows:", consumer_sentiment.duplicated().sum())
print("Missing values:", consumer_sentiment.isnull().sum().sum())

# reset index
consumer_sentiment.reset_index(drop=True, inplace=True)

# check first rows
print(consumer_sentiment.head(5))

# save cleaned csv file
consumer_sentiment.to_csv("../data/cleaned/consumer_sentiment_cleaned.csv", index=False)

Duplicate rows: 0
Missing values: 0
        Date  ConsumerSentiment
0 2007-01-01               96.9
1 2007-02-01               91.3
2 2007-03-01               88.4
3 2007-04-01               87.1
4 2007-05-01               88.3


In [3]:
# read in cpi.csv
cpi = pd.read_csv('../data/raw/cpi.csv')

# rename columns
cpi.columns = ["Date", "CPI"]

# convert to datetime
cpi["Date"] = pd.to_datetime(cpi["Date"])

# filter data for project parameters
cpi = cpi[(cpi["Date"] >= "2007-01-01") &
          (cpi["Date"] <= "2026-05-01")]

# reset index
cpi.reset_index(drop=True, inplace=True)

# check for null and duplicate values
print(cpi.isnull().sum())

# find row with missing value
rows_with_nan = cpi[cpi["CPI"].isna()]
print(rows_with_nan) # october 2025 was not collected due to federal government shutdown

# fill in with linear interpolation
cpi["CPI"] = cpi["CPI"].interpolate()

# check first rows
print(cpi.head(15))

# save cleaned csv file
cpi.to_csv("../data/cleaned/cpi_cleaned.csv", index=False)

Date    0
CPI     1
dtype: int64
          Date  CPI
225 2025-10-01  NaN
         Date      CPI
0  2007-01-01  203.437
1  2007-02-01  204.226
2  2007-03-01  205.288
3  2007-04-01  205.904
4  2007-05-01  206.755
5  2007-06-01  207.234
6  2007-07-01  207.603
7  2007-08-01  207.667
8  2007-09-01  208.547
9  2007-10-01  209.190
10 2007-11-01  210.834
11 2007-12-01  211.445
12 2008-01-01  212.174
13 2008-02-01  212.687
14 2008-03-01  213.448


In [4]:
# read in gas_prices.csv
gas = pd.read_csv('../data/raw/gas_prices.csv')

# rename columns
gas.columns = ["Date", "GasPrices"]

# convert to datetime
gas["Date"] = pd.to_datetime(gas["Date"])

# convert weekly data set to monthly
# first make date the index
gas = gas.set_index("Date")

# regroup weekly data by months
gas = gas.resample("MS")

# calculate monthly average
gas = gas.mean().round(3)

# make Date a normal column
gas = gas.reset_index()

# filter data for project parameters
gas = gas[(gas["Date"] >= "2007-01-01") &
          (gas["Date"] <= "2026-05-01")]

# reset index after filter
gas = gas.reset_index(drop=True)

# check for missing values
print("Missing values:", gas.isnull().sum().sum())

# check first rows
print(gas.head(5))

# save cleaned csv file
gas.to_csv("../data/cleaned/gas_prices_cleaned.csv", index=False)

Missing values: 0
        Date  GasPrices
0 2007-01-01      2.240
1 2007-02-01      2.278
2 2007-03-01      2.563
3 2007-04-01      2.845
4 2007-05-01      3.146


In [5]:
# read in mortgage_rates.csv
mortgage = pd.read_csv('../data/raw/mortgage_rates.csv')

# rename columns
mortgage.columns = ["Date", "MortgageRates"]

# convert the date column
mortgage["Date"] = pd.to_datetime(mortgage["Date"])

# convert weekly data set to monthly
# make date the index
mortgage = mortgage.set_index("Date")

# regroup weekly data into monthly sets
mortgage = mortgage.resample("MS")

# calculate monthly average
mortgage = mortgage.mean().round(2)

# make Date a normal column
mortgage = mortgage.reset_index()

# filter data for project parameters
mortgage = mortgage[(mortgage["Date"] >= "2007-01-01") &
                    (mortgage["Date"] <= "2026-05-01")]

# reset index after filter
mortgage = mortgage.reset_index(drop=True)

# check missing values
print("Missing values:", mortgage.isnull().sum().sum())

# check first rows
print(mortgage.head(5))

# save cleaned csv file
mortgage.to_csv("../data/cleaned/mortgage_rates_cleaned.csv", index=False)

Missing values: 0
        Date  MortgageRates
0 2007-01-01           6.22
1 2007-02-01           6.28
2 2007-03-01           6.16
3 2007-04-01           6.18
4 2007-05-01           6.26


In [6]:
# read in unemployment.csv
unemployment = pd.read_csv("../data/raw/unemployment.csv")

# rename columns
unemployment.columns = ["Date", "UnemploymentRate"]

# convert the date column
unemployment["Date"] = pd.to_datetime(unemployment["Date"])

# filter project period
unemployment = unemployment[(unemployment["Date"] >= "2007-01-01") &
                            (unemployment["Date"] <= "2026-05-01")]

# reset index
unemployment.reset_index(drop=True, inplace=True)

# check missing values
print("Missing values:", unemployment.isnull().sum().sum())

# find row with missing value
rows_with_nan = unemployment[unemployment.isna().any(axis=1)]
print(rows_with_nan)

# use linear interpolation to fill in missing value
unemployment["UnemploymentRate"] = unemployment["UnemploymentRate"].interpolate()

# check first rows
print(unemployment.head(5))

# save cleaned csv file
unemployment.to_csv("../data/cleaned/unemployment_cleaned.csv", index=False)

Missing values: 1
          Date  UnemploymentRate
225 2025-10-01               NaN
        Date  UnemploymentRate
0 2007-01-01               4.6
1 2007-02-01               4.5
2 2007-03-01               4.4
3 2007-04-01               4.5
4 2007-05-01               4.4


In [7]:
# read in EFFR.csv (Effective Federal Funds Rate)
effr = pd.read_csv('../data/raw/EFFR.csv')

# rename columns
effr.columns = ["Date", "FedFundsRate"]

# convert to date
effr["Date"] = pd.to_datetime(effr["Date"])

# convert data to monthly
effr = (effr.set_index("Date")
            .resample("MS")
            .mean().round(2)
            .reset_index())

# filter project period
effr = effr[(effr["Date"] >= "2007-01-01") &
            (effr["Date"] <= "2026-05-01")]

# reset index
effr.reset_index(drop=True, inplace=True)

# check for missing values
print("Missing values:", effr.isnull().sum().sum())

# check first rows
print(effr.head(5))

# save cleaned csv file
effr.to_csv("../data/cleaned/EFFR_cleaned.csv", index=False)

Missing values: 0
        Date  FedFundsRate
0 2007-01-01          5.26
1 2007-02-01          5.27
2 2007-03-01          5.26
3 2007-04-01          5.24
4 2007-05-01          5.25


In [8]:
# read in real_dpi.csv (Real Disposable Personal Income)
income = pd.read_csv("../data/raw/real_dpi.csv")

# rename columns
income.columns = ["Date", "DisposableIncome"]

# convert date column
income["Date"] = pd.to_datetime(income["Date"])

# filter project period
income = income[(income["Date"] >= "2007-01-01") &
                (income["Date"] <= "2026-05-01")]

# reset index
income.reset_index(drop=True, inplace=True)

# verify no missing values
print("Missing values:", income.isnull().sum().sum())

# check first rows
print(income.head(5))

# save cleaned csv file
income.to_csv("../data/cleaned/real_dpi_cleaned.csv", index=False)

Missing values: 0
        Date  DisposableIncome
0 2007-01-01             39803
1 2007-02-01             39906
2 2007-03-01             40007
3 2007-04-01             40037
4 2007-05-01             40029


In [9]:
# read in real_pce.csv (Real Personal Consumption Expenditures)
spending = pd.read_csv("../data/raw/real_pce.csv")

# rename columns
spending.columns = ["Date", "ConsumerSpending"]

# convert date column
spending["Date"] = pd.to_datetime(spending["Date"])

# filter data for project parameters
spending = spending[(spending["Date"] >= "2007-01-01") &
                    (spending["Date"] <= "2026-05-01")]

# reset index
spending.reset_index(drop=True, inplace=True)

# check for missing values
print("Missing values:", spending.isnull().sum().sum())

# check first rows
print(spending.head(5))

# save cleaned csv file
spending.to_csv("../data/cleaned/real_pce_cleaned.csv", index=False)

Missing values: 0
        Date  ConsumerSpending
0 2007-01-01           11181.0
1 2007-02-01           11178.2
2 2007-03-01           11190.7
3 2007-04-01           11201.5
4 2007-05-01           11218.0


In [10]:
# read in hourly_wages.csv (Average Hourly Earnings of All Employees, Total Private)
wage_growth = pd.read_csv("../data/raw/hourly_wages.csv")

# rename columns
wage_growth.columns = ["Date", "HourlyWages"]

# convert date column
wage_growth["Date"] = pd.to_datetime(wage_growth["Date"])

# filter project period
wage_growth = wage_growth[(wage_growth["Date"] >= "2007-01-01") &
                          (wage_growth["Date"] <= "2026-05-01")]

# reset index
wage_growth.reset_index(drop=True, inplace=True)

# check for missing values
print("Missing values:", wage_growth.isnull().sum().sum())

# check first rows
print(wage_growth.head(5))

# save cleaned csv file
wage_growth.to_csv("../data/cleaned/hourly_wages_cleaned.csv", index=False)

Missing values: 0
        Date  HourlyWages
0 2007-01-01        20.59
1 2007-02-01        20.68
2 2007-03-01        20.72
3 2007-04-01        20.79
4 2007-05-01        20.84


In [11]:
savings_rate = pd.read_csv("../data/raw/personal_savings_rate.csv")

savings_rate.columns = ["Date", "SavingsRate"]

savings_rate["Date"] = pd.to_datetime(savings_rate["Date"])

savings_rate = savings_rate[(savings_rate["Date"] >= "2007-01-01") &
                            (savings_rate["Date"] <= "2026-05-01")]

savings_rate.reset_index(drop=True, inplace=True)

print("Missing values:", savings_rate.isnull().sum().sum())

print(savings_rate.head(5))

savings_rate.to_csv("../data/cleaned/savings_rate_cleaned.csv", index=False)

Missing values: 0
        Date  SavingsRate
0 2007-01-01          2.4
1 2007-02-01          2.8
2 2007-03-01          3.0
3 2007-04-01          3.0
4 2007-05-01          2.8
